In [1]:
# import libraries
import pandas as pd
import numpy as np
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

In [2]:
# load csv
macro = pd.read_csv("macro_daily_with_regimes.csv")
macro["session_date"] = pd.to_datetime(macro["session_date"]).dt.date

print(macro.shape)
print(macro.columns)

macro.head()

(2037, 31)
Index(['symbol', 'session_date', 'day_open', 'day_close', 'day_high',
       'day_low', 'day_volume', 'bars_in_day', 'daily_rv', 'daily_ret',
       'daily_abs_ret', 'daily_range_pct', 'daily_trend_strength', 'rv_3',
       'rv_long', 'rv_impulse', 'range_3', 'range_long', 'range_ratio',
       'trend_freq', 'sum_ret_3', 'sum_abs_ret_3', 'conditions_3', 'rel_vol',
       'rv_impulse_z', 'range_ratio_z', 'trend_freq_z', 'conditions_3_z',
       'rel_vol_z', 'regime_label', 'macro_regime_id'],
      dtype='str')


,symbol,session_date,day_open,day_close,day_high,day_low,day_volume,bars_in_day,daily_rv,daily_ret,...,sum_abs_ret_3,conditions_3,rel_vol,rv_impulse_z,range_ratio_z,trend_freq_z,conditions_3_z,rel_vol_z,regime_label,macro_regime_id
0,COIN,2024-04-25,216.25,223.61,225.94,213.64,4451838.0,78,0.005675,0.034035,...,0.153423,0.310310,-0.359211,-1.107358,-1.094029,1.760328,-0.708067,-1.163974,Range Bound,3
1,COIN,2024-04-26,220.77,236.44,237.02,218.66,5472576.0,78,0.004153,0.070979,...,0.144204,0.266219,-0.592937,-1.152497,-0.886615,0.583332,-0.856829,-2.086605,Range Bound,3
2,COIN,2024-04-29,229.94,218.14,230.32,216.54,8487485.0,78,0.005703,-0.051318,...,0.157921,0.329953,-0.367734,-0.749050,-0.528739,0.548018,-0.610722,-0.997723,Range Bound,3
3,COIN,2024-04-30,214.67,204.02,216.57,202.59,7891047.0,78,0.004589,-0.049611,...,0.156331,0.343474,0.071109,-0.184412,-0.157973,0.497989,-0.519229,0.762565,Range Bound,3
4,COIN,2024-05-01,199.00,209.93,218.52,198.20,8957379.0,78,0.005568,0.054925,...,0.171908,0.174221,0.033964,-0.627555,0.257219,1.532549,-1.067845,0.575629,Range Bound,3


In [3]:
# define features
macro_feature_cols = [
    "rv_impulse","range_ratio","trend_freq","conditions_3","rel_vol",
    "daily_rv","daily_range_pct","day_volume","daily_ret","daily_abs_ret","daily_trend_strength"
]

REGIME_INT    = {0: "Flat", 1: "Trending", 2: "Choppy", 3: "Range Bound"}
target_col    = "macro_regime_id"

print(macro.shape)
print(macro["regime_label"].value_counts())

(2037, 31)
regime_label
Range Bound    873
Trending       710
Choppy         332
Flat           122
Name: count, dtype: int64


In [4]:
# target column
# regime id (0=Flat, 1=Trending, 2=Choppy, 3=Range Bound)
target_col = "macro_regime_id"

In [5]:
# split data based on dates
first_val_day  = pd.to_datetime("2025-07-17").date()
first_test_day = pd.to_datetime("2025-10-29").date()

macro = macro.sort_values(["symbol", "session_date"]).reset_index(drop=True)

train_mask = macro["session_date"] < first_val_day

# scale data
scaler = StandardScaler()
scaler.fit(macro.loc[train_mask, macro_feature_cols])
macro[macro_feature_cols] = scaler.transform(macro[macro_feature_cols])
joblib.dump(scaler, "transformer_daily_scaler.pkl")

print("Train rows:", train_mask.sum())
print("Val rows:  ", ((macro["session_date"] >= first_val_day) & (macro["session_date"] < first_test_day)).sum())
print("Test rows: ", (macro["session_date"] >= first_test_day).sum())

Train rows: 1359
Val rows:   330
Test rows:  348


In [6]:
# build sequence dataset using 3 days worth of price data
LOOKBACK_DAYS = 3

def build_daily_sequences(df, feature_cols, target_col, lookback):
    X_list, y_list, meta_rows = [], [], []

    for symbol, g in df.groupby("symbol"):
        g = g.sort_values("session_date").reset_index(drop=True)
        days = g["session_date"].values

        for i in range(lookback, len(g)):
            X_seq = g.iloc[i-lookback:i][feature_cols].values.astype(np.float32)
            y_label = int(g.iloc[i][target_col])
            X_list.append(X_seq)
            y_list.append(y_label)
            meta_rows.append((symbol, days[i]))

    X   = np.array(X_list, dtype=np.float32)
    y   = np.array(y_list, dtype=np.int64)
    meta = pd.DataFrame(meta_rows, columns=["symbol", "session_date"])
    return X, y, meta

In [7]:
# build sequences
X_all, y_all, meta_all = build_daily_sequences(macro, macro_feature_cols, target_col, LOOKBACK_DAYS)
meta_all["session_date"] = pd.to_datetime(meta_all["session_date"]).dt.date

# split data by date
train_mask = meta_all["session_date"] < first_val_day
val_mask   = (meta_all["session_date"] >= first_val_day) & (meta_all["session_date"] < first_test_day)
test_mask  = meta_all["session_date"] >= first_test_day

X_train, y_train, meta_train = X_all[train_mask], y_all[train_mask], meta_all[train_mask].reset_index(drop=True)
X_val,   y_val,   meta_val   = X_all[val_mask],   y_all[val_mask],   meta_all[val_mask].reset_index(drop=True)
X_test,  y_test,  meta_test  = X_all[test_mask],  y_all[test_mask],  meta_all[test_mask].reset_index(drop=True)

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape,   y_val.shape)
print("Test: ", X_test.shape,  y_test.shape)

Train: (1344, 3, 11) (1344,)
Val:   (330, 3, 11) (330,)
Test:  (348, 3, 11) (348,)


In [8]:
# check class balance
for split_name, y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    counts = pd.Series(y).value_counts().sort_index().rename(REGIME_INT)
    print(f"\n{split_name}:")
    print(counts)


Train:
Flat            95
Trending       456
Choppy         230
Range Bound    563
Name: count, dtype: int64

Val:
Flat             5
Trending       116
Choppy          54
Range Bound    155
Name: count, dtype: int64

Test:
Flat            22
Trending       136
Choppy          44
Range Bound    146
Name: count, dtype: int64


In [9]:
# create pytorch datasets
class RegimeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):        return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

In [10]:
# create dataloaders
BATCH_SIZE = 32
train_loader = DataLoader(RegimeDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(RegimeDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(RegimeDataset(X_test,  y_test),  batch_size=BATCH_SIZE, shuffle=False)

### Transformer Model

In [11]:
# create model
class RegimeTransformer(nn.Module):
    def __init__(self, num_features, num_classes, seq_len=3,
                 d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.2):
        super().__init__()
        self.input_proj    = nn.Linear(num_features, d_model)
        self.pos_embedding = nn.Parameter(torch.zeros(1, seq_len, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.encoder    = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm       = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.input_proj(x) + self.pos_embedding
        x = self.encoder(x)
        x = self.norm(x.mean(dim=1))
        return self.classifier(x)

In [12]:
# instantiate model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = RegimeTransformer(
    num_features=len(macro_feature_cols),
    num_classes=4,
    seq_len=LOOKBACK_DAYS
).to(device)

print(model)
print(f"\nDevice: {device}")

RegimeTransformer(
  (input_proj): Linear(in_features=11, out_features=64, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (classifier): Linear(in_features=64, out_features=4, bias=True)
)

Device: cuda


In [13]:
# apply class weights
class_counts = np.bincount(y_train)

class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print(class_weights)

tensor([2.2345, 0.4655, 0.9229, 0.3770], device='cuda:0')


In [14]:
# set loss function and optimizer
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

In [15]:
# train and validation
def train_epoch(model, loader, optimizer, criterion, device):

    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset)

def eval_epoch(model, loader, criterion, device):

    model.eval()
    total_loss = 0
    correct = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == yb).sum().item()
    loss = total_loss / len(loader.dataset)
    acc = correct / len(loader.dataset)
    return loss, acc

In [16]:
# train with early stopping
EPOCHS = 100
PATIENCE = 12

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

best_val_loss = float("inf")
patience_counter = 0

train_losses = []
val_losses = []
val_accs = []

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion, device)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val Acc:    {val_acc:.4f}")
    print()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "transformer_regime_model.pt")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered")
            break

print(f"\nBest val loss: {best_val_loss:.4f}")

Epoch 1
Train Loss: 1.2978
Val Loss:   1.3121
Val Acc:    0.4061

Epoch 2
Train Loss: 1.1793
Val Loss:   1.2615
Val Acc:    0.3879

Epoch 3
Train Loss: 1.1269
Val Loss:   1.2590
Val Acc:    0.3939

Epoch 4
Train Loss: 1.0930
Val Loss:   1.2320
Val Acc:    0.3939

Epoch 5
Train Loss: 1.0630
Val Loss:   1.1600
Val Acc:    0.4636

Epoch 6
Train Loss: 1.0187
Val Loss:   1.1112
Val Acc:    0.4364

Epoch 7
Train Loss: 0.9843
Val Loss:   1.1387
Val Acc:    0.4818

Epoch 8
Train Loss: 0.9091
Val Loss:   1.0322
Val Acc:    0.5485

Epoch 9
Train Loss: 0.8496
Val Loss:   0.9725
Val Acc:    0.5000

Epoch 10
Train Loss: 0.7841
Val Loss:   0.9173
Val Acc:    0.5879

Epoch 11
Train Loss: 0.7368
Val Loss:   0.8725
Val Acc:    0.5818

Epoch 12
Train Loss: 0.6975
Val Loss:   0.7394
Val Acc:    0.6394

Epoch 13
Train Loss: 0.7124
Val Loss:   0.7691
Val Acc:    0.6424

Epoch 14
Train Loss: 0.6568
Val Loss:   0.7864
Val Acc:    0.5970

Epoch 15
Train Loss: 0.6455
Val Loss:   0.7215
Val Acc:    0.6455

Epoc

In [17]:
# load best model
model.load_state_dict(torch.load("transformer_regime_model.pt"))
model.eval()

RegimeTransformer(
  (input_proj): Linear(in_features=11, out_features=64, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (classifier): Linear(in_features=64, out_features=4, bias=True)
)

In [18]:
# test model against test dataset
def get_predictions_with_probs(model, loader, meta, device):

    model.eval()
    all_probs = []
    all_preds = []
    all_true  = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_preds.extend(np.argmax(probs, axis=1))
            all_true.extend(yb.numpy())
    probs_arr = np.vstack(all_probs)
    result = meta.copy().reset_index(drop=True)
    result["true_regime_id"]   = all_true
    result["pred_regime_id"]   = all_preds
    result["prob_flat"]        = probs_arr[:, 0]
    result["prob_trending"]    = probs_arr[:, 1]
    result["prob_choppy"]      = probs_arr[:, 2]
    result["prob_range_bound"] = probs_arr[:, 3]
    return result

model.load_state_dict(torch.load("transformer_regime_model.pt"))

train_preds_df = get_predictions_with_probs(model, train_loader, meta_train, device)
val_preds_df   = get_predictions_with_probs(model, val_loader,   meta_val,   device)
test_preds_df  = get_predictions_with_probs(model, test_loader,  meta_test,  device)

train_preds_df["split"] = "train"
val_preds_df["split"]   = "val"
test_preds_df["split"]  = "test"

all_preds_df = pd.concat([train_preds_df, val_preds_df, test_preds_df], ignore_index=True)
all_preds_df.to_csv("transformer_regime_predictions_all.csv", index=False)
print("Saved transformer_regime_predictions_all.csv")
print(all_preds_df.head())

Saved transformer_regime_predictions_all.csv
  symbol session_date  true_regime_id  pred_regime_id  prob_flat  \
0   COIN   2024-04-30               0               0   0.923261   
1   COIN   2024-05-01               3               2   0.014593   
2   COIN   2024-05-02               3               3   0.008746   
3   COIN   2024-05-03               2               2   0.008251   
4   COIN   2024-05-06               3               2   0.010370   

   prob_trending  prob_choppy  prob_range_bound  split  
0       0.024275     0.001191          0.051273  train  
1       0.001444     0.669335          0.314629  train  
2       0.038445     0.323388          0.629420  train  
3       0.000478     0.872173          0.119098  train  
4       0.000892     0.798698          0.190040  train  


In [19]:
# create classification report
target_names = ["Flat", "Trending", "Choppy", "Range Bound"]
print(classification_report(
    test_preds_df["true_regime_id"],
    test_preds_df["pred_regime_id"],
    target_names=target_names
))

              precision    recall  f1-score   support

        Flat       0.36      0.59      0.45        22
    Trending       0.85      0.93      0.89       136
      Choppy       0.63      0.61      0.62        44
 Range Bound       0.79      0.65      0.71       146

    accuracy                           0.75       348
   macro avg       0.66      0.70      0.67       348
weighted avg       0.77      0.75      0.75       348



In [20]:
# add predictions to entries
entries_df = pd.read_csv("multiasset_labeled_entries_with_regime.csv")
entries_df["session_date"] = pd.to_datetime(entries_df["ts_entry"]).dt.date.astype(str)
all_preds_df["session_date"] = all_preds_df["session_date"].astype(str)

prob_cols = ["pred_regime_id", "prob_flat", "prob_trending", "prob_choppy", "prob_range_bound"]
entries_with_preds = entries_df.merge(
    all_preds_df[["symbol", "session_date"] + prob_cols],
    on=["symbol", "session_date"],
    how="left"
)

print("Total entries:", len(entries_with_preds))
print("Missing pred_regime_id:", entries_with_preds["pred_regime_id"].isna().sum())
print("\nPredicted regime distribution:")
print(entries_with_preds["pred_regime_id"].map(REGIME_INT).value_counts())

entries_with_preds.to_csv("multiasset_labeled_entries_with_transformer_preds.csv", index=False)
print("\nSaved multiasset_labeled_entries_with_transformer_preds.csv")

Total entries: 1464
Missing pred_regime_id: 29

Predicted regime distribution:
pred_regime_id
Trending       533
Range Bound    468
Choppy         295
Flat           139
Name: count, dtype: int64

Saved multiasset_labeled_entries_with_transformer_preds.csv
